In [ ]:
%run manage_files

In [ ]:
from pyspark.sql.functions import *

In [ ]:
class TaxiFolder:

    def __init__(self, folder_path, table_name):
        self.folder_path = folder_path
        self.table_name = table_name
    
    def __get_existing_months(self, df, date_column):
        return [row["yyyy-mm"] for row in df.select(date_format(col(date_column), "yyyy-MM").alias("yyyy-mm")).distinct().collect()]

    def __get_loaded_months(self, date_column):
        if not is_table_exist(self.table_name, "Silver_lakehouse"):
            return []
        df = read_table(self.table_name, "Silver_lakehouse")
        return self.__get_existing_months(df, date_column)

    def __get_new_files(self, file_paths, existing_months):
        return [path for path in file_paths if path[-15:-8] not in existing_months]

    def get_unloaded_parquet_files(self, date_column):
        file_paths = get_parquet_files(self.folder_path)
        existing_months = self.__get_loaded_months(date_column)
        return self.__get_new_files(file_paths, existing_months)